In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [ ]:
data = pd.read_csv('data.csv')
data.head(4)

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_and_lemmatize(text):
    if isinstance(text, str):

        tokens = word_tokenize(text.lower())

        tokens = [token for token in tokens if token.isalnum() and token not in stop_words]

        tokens = [lemmatizer.lemmatize(token) for token in tokens]

        return ' '.join(tokens)
    else:
        return ''

data['cleaned_text'] = data['email'].apply(clean_and_lemmatize)

data.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.decomposition import TruncatedSVD


tfidf_vectorizer = TfidfVectorizer(max_features=500)
tfidf_features = tfidf_vectorizer.fit_transform(data['cleaned_text'])

print("TF-IDF Features:")
print(tfidf_features.toarray())



sentences = [text.split() for text in data['cleaned_text'].tolist()]
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def document_vector(doc):
    doc_vec = np.zeros(100)
    word_count = 0
    for word in doc:
        if word in word2vec_model.wv:
            doc_vec += word2vec_model.wv[word]
            word_count += 1
    if word_count > 0:
        doc_vec /= word_count
    return doc_vec

word2vec_features = np.array([document_vector(doc) for doc in sentences])

print("\nWord2Vec Features:")
print(word2vec_features)

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



X = word2vec_features
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_lstm = Sequential()
model_lstm.add(LSTM(128, input_shape=(X_train.shape[1], 1)))
model_lstm.add(Dense(1, activation='sigmoid'))

model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model_lstm.fit(np.expand_dims(X_train, axis=2), y_train, epochs=10, batch_size=32, validation_split=0.1)

loss, accuracy = model_lstm.evaluate(np.expand_dims(X_test, axis=2), y_test)
print("LSTM Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_lstm = (model_lstm.predict(np.expand_dims(X_test, axis=2)) > 0.5).astype("int32")
def evaluate_model(y_true, y_pred, model_name):
  f1 = f1_score(y_true, y_pred)
  accuracy = accuracy_score(y_true, y_pred)
  precision = precision_score(y_true, y_pred)
  print(f"{model_name} Metrics:")
  print(f"  F1 Score: {f1}")
  print(f"  Accuracy: {accuracy}")
  print(f"  Precision: {precision}")
  return f1, accuracy, precision

f1_lstm, accuracy_lstm, precision_lstm = evaluate_model(y_test, y_pred_lstm, "LSTM")

models = ["LSTM"]
f1_scores = [f1_lstm]
accuracy_scores = [accuracy_lstm]
precision_scores = [precision_lstm]

x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width, f1_scores, width, label='F1 Score')
rects2 = ax.bar(x, accuracy_scores, width, label='Accuracy')
rects3 = ax.bar(x + width, precision_scores, width, label='Precision')

ax.set_ylabel('Scores')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()

plt.show()

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm_lstm = confusion_matrix(y_test, y_pred_lstm)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('LSTM Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get the predicted probabilities for LSTM and XLSTM (not just binary predictions)
y_pred_prob_lstm = model_lstm.predict(np.expand_dims(X_test, axis=2))
y_pred_prob_xlstm = model_xlstm.predict(np.expand_dims(X_test, axis=2))

# Compute ROC curve and AUC for LSTM
fpr_lstm, tpr_lstm, _ = roc_curve(y_test, y_pred_prob_lstm)
roc_auc_lstm = auc(fpr_lstm, tpr_lstm)

# Compute ROC curve and AUC for XLSTM
fpr_xlstm, tpr_xlstm, _ = roc_curve(y_test, y_pred_prob_xlstm)
roc_auc_xlstm = auc(fpr_xlstm, tpr_xlstm)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr_lstm, tpr_lstm, color='blue', lw=2, label=f'LSTM (AUC = {roc_auc_lstm:.2f})')
plt.plot(fpr_xlstm, tpr_xlstm, color='green', lw=2, label=f'XLSTM (AUC = {roc_auc_xlstm:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')  # Diagonal line (random classifier)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
